In [1]:
import os

# Merge netcdf files with gluemncbig

In [2]:
path_results_folder = r"D:\geneva_nc_10min"

# Define paths
output_file = os.path.join(path_results_folder, 'grid_merged.nc')
input_files = os.path.join(path_results_folder, r'grid.*.nc')

# Run the command
!python C:\Users\leroquan\Documents\python\MITgcm\utils\python\MITgcmutils\scripts\gluemncbig -o "{output_file}" "{input_files}" --many

In [ ]:
# Define paths
output_file = os.path.join(path_results_folder, '3Dsnaps_merged.nc')
input_files = os.path.join(path_results_folder, r'3Dsnaps.*.nc')

# Run the command
!python C:\Users\leroquan\Documents\python\MITgcm\utils\python\MITgcmutils\scripts\gluemncbig -o "{output_file}" "{input_files}" --many

# Visualise glued results

In [1]:
import xarray as xr

In [4]:
ds = xr.open_dataset(r"D:\geneva_nc_10min\3Dsnaps_merged.nc")

In [3]:
xr.open_dataset(r"D:\geneva_nc_10min\grid_merged.nc")

<xarray.Dataset> Size: 114MB
Dimensions:  (Z: 100, Zp1: 101, Zu: 100, Zl: 100, X: 336, Y: 132, Xp1: 337,
              Yp1: 133)
Coordinates:
  * Z        (Z) float64 800B -0.25 -0.7575 -1.28 -1.82 ... -295.6 -305.2 -315.1
  * Zp1      (Zp1) float64 808B 0.0 -0.5 -1.015 -1.546 ... -300.4 -310.1 -320.1
  * Zu       (Zu) float64 800B -0.5 -1.015 -1.546 ... -300.4 -310.1 -320.1
  * Zl       (Zl) float64 800B 0.0 -0.5 -1.015 -1.546 ... -290.9 -300.4 -310.1
  * X        (X) float64 3kB 100.0 300.0 500.0 ... 6.67e+04 6.69e+04 6.71e+04
  * Y        (Y) float64 1kB 100.0 300.0 500.0 ... 2.59e+04 2.61e+04 2.63e+04
  * Xp1      (Xp1) float64 3kB 0.0 200.0 400.0 ... 6.68e+04 6.7e+04 6.72e+04
  * Yp1      (Yp1) float64 1kB 0.0 200.0 400.0 ... 2.6e+04 2.62e+04 2.64e+04
Data variables: (12/30)
    RC       (Z) float64 800B ...
    RF       (Zp1) float64 808B ...
    RU       (Zu) float64 800B ...
    RL       (Zl) float64 800B ...
    drC      (Zp1) float64 808B ...
    drF      (Z) float64 800B ...
    ...       ...
    R_low    (Y, X) float64 355kB ...
    Ro_surf  (Y, X) float64 355kB ...
    Depth    (Y, X) float64 355kB ...
    HFacC    (Z, Y, X) float64 35MB ...
    HFacW    (Z, Y, Xp1) float64 36MB ...
    HFacS    (Z, Yp1, X) float64 36MB ...
Attributes: (12/18)
    MITgcm_version:  checkpoint67z
    build_user:      aleroqua
    build_host:      daint-ln003
    build_date:      Tue 29 Jul 2025 05:48:07 PM CEST
    MITgcm_URL:      http://mitgcm.org
    MITgcm_tag_id:   
    ...              ...
    nSy:             1
    nPx:             48
    nPy:             12
    Nx:              336
    Ny:              132
    Nr:              100

# Using James' code from Alplakes

In [27]:
import os
import glob
from datetime import datetime
import numpy as np
import pandas as pd
import json
import xarray as xr
import netCDF4

In [24]:
class MitgcmGrid:
    """Class representing an MITgcm grid, with optional loading from .npy files."""

    def __init__(self):
        """Initialize an empty MITgcm grid."""
        self.x = np.array([])
        self.y = np.array([])
        self.lat_grid = np.array([])
        self.lon_grid = np.array([])
        self.dz = np.array([])
        self.parameters = {}

    def load_from_path(self, path_grid: str):
        """
        Load grid data from a given folder containing .npy files.

        Args:
            path_grid (str): Path to the folder containing the grid files.

        Raises:
            FileNotFoundError: If any required grid file is missing.
            RuntimeError: If loading fails due to other errors.
        """
        try:
            self.x = np.load(os.path.join(path_grid, 'x.npy'))
            self.y = np.load(os.path.join(path_grid, 'y.npy'))
            self.lat_grid = np.load(os.path.join(path_grid, 'lat_grid.npy'))
            self.lon_grid = np.load(os.path.join(path_grid, 'lon_grid.npy'))
            self.dz = pd.read_csv(os.path.join(path_grid, 'dz.csv'), header=None).to_numpy()
            with open(os.path.join(path_grid, 'parameters.json'), 'r') as file:
                self.parameters = json.load(file)
        except FileNotFoundError as e:
            raise FileNotFoundError(f"Missing grid file: {e.filename}") from e
        except Exception as e:
            raise RuntimeError(f"Error loading grid data: {e}") from e

In [25]:
nc_folder_path = r'D:\geneva_nc_10min'
nodata = np.nan

output_files = glob.glob(os.path.join(nc_folder_path, "3Dsnaps*"))
output_files.sort()

ds_grid = xr.open_dataset(os.path.join(nc_folder_path, 'grid_merged.nc'))

ds0 = xr.open_mfdataset(output_files[0])

In [29]:
grid = MitgcmGrid()
grid.load_from_path(r'C:\Users\leroquan\Documents\Data\mitgcm_grids\geneva\200m')

In [22]:
full_time = ds0["T"].values
general_attributes = ds0.attrs

dimensions = {
    'time': {'dim_name': 'time', 'dim_size': None},
    'Z': {'dim_name': 'depth', 'dim_size': len(ds_grid.Z)},
    'XC': {'dim_name': 'X', 'dim_size': len(ds_grid.X)},
    'YC': {'dim_name': 'Y', 'dim_size': len(ds_grid.Y)}
}
variables = {
    'time': {'var_name': 'time', 'dim': ('time',), 'unit': 'seconds since 1970-01-01 00:00:00',
             'long_name': 'time'},
    'depth': {'var_name': 'depth', 'dim': ('Z',), 'unit': 'm', 'long_name': 'Depth below surface'},
    'lat': {'var_name': 'lat', 'dim': ('YC', 'XC',), 'unit': '', 'long_name': 'Latitude'},
    'lng': {'var_name': 'lng', 'dim': ('YC', 'XC',), 'unit': '', 'long_name': 'Longitude'},
    'THETA': {'var_name': 'temperature', 'dim': ('time', 'Z', 'YC', 'XC',), 'unit': '°C', 'long_name': 'Temperature'},
    'UVEL': {'var_name': 'uvel', 'dim': ('time', 'Z', 'YC', 'XC',), 'unit': 'm/s', 'long_name': 'Eastward velocity'},
    'VVEL': {'var_name': 'vvel', 'dim': ('time', 'Z', 'YC', 'XC',), 'unit': 'm/s', 'long_name': 'Northward velocity'},
    'WVEL': {'var_name': 'wvel', 'dim': ('time', 'Z', 'YC', 'XC',), 'unit': 'm/s', 'long_name': 'Vertical velocity'},
}

In [ ]:
with netCDF4.Dataset(os.path.join(nc_folder_path, "3Dsnaps_merged.nc"), "w") as dst:
    for key in general_attributes:
        setattr(dst, key, general_attributes[key])
    for key, values in dimensions.items():
        dst.createDimension(values['dim_name'], values['dim_size'])
    for key, values in variables.items():
        variables[key]["nc"] = dst.createVariable(values["var_name"], np.float64, values["dim"], fill_value=nodata)
        variables[key]["nc"].units = values["unit"]
        variables[key]["nc"].long_name = values["long_name"]

    variables["time"]["nc"][:] = full_time
    variables["depth"]["nc"][:] = depth
    variables["lat"]["nc"][:] = grid.lat_grid
    variables["lng"]["nc"][:] = grid.lon_grid

    for f in output_files:
        print("  Reading {}".format(os.path.basename(os.path.dirname(f[0]))))
        if len(f) > 1:
            with xr.open_mfdataset(f) as ds:
                x = np.array(ds["X"].values)
                y = ds["Y"].values
                t = ds["THETA"].isel(T=slice(indices[0], indices[-1] + 1)).values
                w = ds["WVEL"].isel(T=slice(indices[0], indices[-1] + 1)).values
                uvel = ds["UVEL"].isel(T=slice(indices[0], indices[-1] + 1)).values
                vvel = ds["VVEL"].isel(T=slice(indices[0], indices[-1] + 1)).values
        else:
            with netCDF4.Dataset(f[0], "r") as nc:
                x = np.array(nc.variables["X"][:])
                y = nc.variables["Y"][:]
                t = nc.variables["THETA"][indices[0]:indices[-1] + 1, :]
                w = nc.variables["WVEL"][indices[0]:indices[-1] + 1, :]
                uvel = nc.variables["UVEL"][indices[0]:indices[-1] + 1, :]
                vvel = nc.variables["VVEL"][indices[0]:indices[-1] + 1, :]

        uvel = (uvel[..., :-1] + uvel[..., 1:]) / 2  # Get cell center
        vvel = (vvel[..., :-1, :] + vvel[..., 1:, :]) / 2 # Get cell center
        if "rotation" in grid.parameters:
            print("    Rotating u,v by {}°".format(-grid.parameters["rotation"]))
            theta_rad = np.deg2rad(-grid.parameters["rotation"])
            u = uvel * np.cos(theta_rad) - vvel * np.sin(theta_rad)
            v = uvel * np.sin(theta_rad) + vvel * np.cos(theta_rad)
        else:
            u = uvel
            v = vvel

        mask = (t == 0.0) | np.isnan(t)
        t[mask] = nodata
        w[mask] = nodata
        u[mask] = nodata
        v[mask] = nodata

        failed = np.all(t == nodata, axis=(1, 2, 3))
        if np.any(failed):
            failed_index = np.argmax(failed)
            print("Simulation failed at time: {}, index: {}".format(time[failed_index], failed_index))
            #os.remove(os.path.join(output_folder, week_start + ".nc"))
            #raise ValueError("Simulation failed at time: {}, index: {}".format(time[failed_index], failed_index))

        data = {"t": t, "w": w, "u": u, "v": v}

        for key, values in data.items():
            variables[key]["nc"][:, :, int(y[0] - 1):int(y[-1]), int(x[0] - 1): int(x[-1])] = values

        print("    Computing thermocline.")
        dt = np.reshape(t, [t.shape[0], t.shape[1], t.shape[2] * t.shape[3]])
        dt[dt == -999] = np.nan
        array = xr.DataArray(
            data=dt,
            dims=["time", "depth", "data"],
            coords=dict(
                time=("time", time),
                depth=("depth", depth),
                data=("data", np.arange(dt.shape[2]))
            )
        )
        therm, index = pylake.thermocline(array)
        therm = np.array(therm)
        therm = np.reshape(therm, [therm.shape[0], t.shape[2], t.shape[3]])
        therm[therm == np.nanmax(therm)] = np.nan
        therm[therm < 0] = np.nan
        therm[therm > np.nanmax(depth)] = np.nan
        therm[np.isnan(therm)] = nodata
        variables["thermocline"]["nc"][:, int(y[0] - 1):int(y[-1]), int(x[0] - 1): int(x[-1])] = therm

pickups = list(set([f.split(".")[1] for f in os.listdir(os.path.join(folder, "run")) if "pickup.00" in f]))
for pickup in pickups:
    with open(os.path.join(folder, "run", "pickup.{}.meta".format(pickup)), "r") as file:
        lines = file.readlines()
    name = False
    for i, line in enumerate(lines):
        if line.strip().startswith("timeStepNumber"):
            lines[i] = " timeStepNumber = [          0 ];\n"
        if line.strip().startswith("timeInterval"):
            dt = origin + timedelta(seconds=float(line.split("=")[1].split("[")[1].split("]")[0].strip()))
            if dt.weekday() != 6 or dt.hour != 0 or dt.minute != 0:
                raise ValueError("Pickup file produced for incorrect date")
            name = dt.strftime("%Y%m%d")
            lines[i] = " timeInterval = [  0.0 ];\n"
    if name:
        shutil.copy(os.path.join(folder, "run", "pickup.{}.data".format(pickup)),
                    os.path.join(folder, "run", "pickup.{}.data".format(name)))
        with open(os.path.join(folder, "run", "pickup.{}.meta".format(name)), "w") as file:
            file.writelines(lines)